# 01 — Ingest

> **AI-Assisted Development** — This project was built with [Kiro](https://kiro.dev). See `README.md` and `SOURCES.md` for full disclosure.

Fetches all raw data for the **dui-by-state** project and loads it into DuckDB.
Raw files are saved to `data/raw/` unchanged. Nothing is modified here.

### Data sources (all U.S. government / official)
| Table | Source | Notes |
|---|---|---|
| `state_ref` | U.S. Census Bureau Gazetteer | Hand-curated CSV — lat/lng, area, region |
| `fips_codes` | U.S. Census Bureau `state.txt` | Official FIPS codes + abbreviations |
| `population` | Census NST-EST2024 | State pop estimates 2020–2024 |
| `fars_trends` | NHTSA FARS 2015–2024 | Alcohol-impaired fatalities by state/year |
| `fars_2024` | NHTSA FARS 2024 | 2024 state summary |
| `dui_laws_ncsl` | NCSL | DUI criminal status by state |
| `dui_penalties_roadlaw` | roadlawguide.com | First-offense penalties (cross-ref) |
| `dui_penalties_ailawyer` | ailawyer.pro | First-offense penalties (cross-ref) |
| `fars_prior_dwi_speed` | NHTSA FARS 2024 | Prior DWI convictions & crash speed |
| `speed_limits_vmt` | IIHS + FHWA | Max speed limits & VMT |
| `fars_bac_testing_2024` | NHTSA FARS 2024 person.csv | BAC testing rates by state (killed/surviving) |
| `bac_testing_laws` | Hand-curated (statutes) | Mandatory vs probable-cause testing laws |
| `nasid_enforcement` | NASID (nasid.org) | 12 enforcement procedure fields per state |
| `states` | Derived | Master reference: all of the above joined |
| `_sources` | Metadata | Provenance record for every table |

Full attribution: see `SOURCES.md`.

---
**To re-run ingestion:** execute `scripts/ingest_all.py` from the project root.
This notebook loads the already-ingested data for inspection and verification.

In [ ]:
import sys
sys.path.insert(0, "..")

import duckdb
import pandas as pd
from pathlib import Path
from src.ingest import load_config

cfg = load_config("../config.yaml")
DB  = Path("..") / cfg["settings"]["duckdb_file"]
con = duckdb.connect(str(DB))

print("Project:", cfg["project_name"])
print("DuckDB: ", DB.resolve())
print()
print("Tables in database:")
con.execute("SHOW TABLES").df()

## Source provenance
Every table has an entry in `_sources` recording where the data came from.

In [ ]:
con.execute("SELECT duckdb_table, source_name, license, retrieved FROM _sources").df()

---
## 1. State reference
Hand-curated from the Census Bureau 2024 Gazetteer.
51 rows (50 states + DC). Includes lat/lng centroids, land area, Census region/division.

In [ ]:
state_ref = con.execute("SELECT * FROM state_ref ORDER BY state_fips").df()
print(f"{len(state_ref)} rows  {list(state_ref.columns)}")
state_ref

---
## 2. Population estimates (Census NST-EST2024)
Annual July 1 estimates per state, 2020–2024.

In [ ]:
pop = con.execute("SELECT * FROM population ORDER BY state_fips").df()
print(f"{len(pop)} rows  {list(pop.columns)}")
pop

---
## 3. FARS — alcohol-impaired fatalities (2015–2024)
Source: NHTSA Fatality Analysis Reporting System.

**Methodology note:**
- 2015–2020: `DRUNK_DR > 0` in `accident.csv` flags alcohol-involved crashes.
- 2021–2024: NHTSA restructured the schema. Alcohol involvement derived from
  `drimpair.csv` where `drimpair = 9` (Under the Influence of Alcohol, Drugs or Medication),
  joined to `accident.csv` on `ST_CASE`.
- The 2021+ figures are lower than 2015–2020 because the new schema captures only
  cases explicitly coded — not all alcohol-involved crashes are coded by officers.
  NHTSA's own published totals (using statistical imputation) are higher.

In [ ]:
# National totals by year
national = con.execute("""
    SELECT
        year,
        SUM(total_fatalities)    AS total_fatalities,
        SUM(impaired_fatalities)  AS impaired_fatalities,
        ROUND(SUM(impaired_fatalities) * 100.0 / SUM(total_fatalities), 1) AS pct_impaired
    FROM fars_trends
    GROUP BY year
    ORDER BY year
""").df()
national

In [ ]:
# 2024 by state — top 15 by impaired fatality %
con.execute("""
    SELECT state_fips, state_name, total_fatalities, impaired_fatalities,
           pct_fatalities_impaired AS pct_impaired
    FROM fars_2024
    ORDER BY pct_fatalities_impaired DESC
    LIMIT 15
""").df()

---
## 4. DUI laws & penalties
Three sources scraped and cross-referenced.
Always verify against official state statutes before publishing.

In [ ]:
# NCSL — DUI criminal status (misdemeanor vs felony thresholds)
ncsl = con.execute("SELECT * FROM dui_laws_ncsl").df()
print(f"NCSL: {len(ncsl)} rows")
ncsl.head(10)

In [ ]:
# roadlawguide — penalties: BAC, fines, jail, suspension, IID
roadlaw = con.execute("SELECT * FROM dui_penalties_roadlaw").df()
print(f"roadlawguide: {len(roadlaw)} rows")
roadlaw.head(10)

In [ ]:
# ailawyer — penalties: jail, fine, suspension, IID, lookback, felony threshold
ail = con.execute("SELECT * FROM dui_penalties_ailawyer").df()
print(f"ailawyer: {len(ail)} rows")
ail.head(10)

---
## 5. Master states table
Joins state reference, population, and FARS 2024 summary into one row per state.

In [ ]:
states = con.execute("SELECT * FROM states ORDER BY state_fips").df()
print(f"{len(states)} rows × {len(states.columns)} cols")
print(f"Columns: {list(states.columns)}")
states

---
## 6. NIAAA Alcohol Consumption by State (2022)
Source: National Institute on Alcohol Abuse and Alcoholism, Surveillance Report #121.

- **URL:** https://www.niaaa.nih.gov/sites/default/files/surveillance-report121.pdf
- **Method:** PDF Table 2 extracted via `pdfplumber`
- **License:** Public domain (U.S. government work)
- **Key field:** `ethanol_per_capita_gallons_2022` — gallons of pure ethanol per person age 14+
- **Decile:** 1 = highest consumption, 10 = lowest. National avg 2.50 gal.

In [ ]:
consumption = con.execute("SELECT * FROM alcohol_consumption ORDER BY ethanol_per_capita_gallons_2022 DESC").df()
print(f"{len(consumption)} states")
consumption

---
## 7. NHTSA Imputed Alcohol Fatalities (2024)
Source: NHTSA Traffic Safety Facts — State Alcohol-Impaired-Driving Estimates, 2024.

- **URL:** https://crashstats.nhtsa.dot.gov/Api/Public/ViewPublication/813813
- **Method:** PDF Table 2 extracted via `pdfplumber`
- **License:** Public domain (U.S. government work)
- **Key difference from `fars_2024`:** These are **statistically imputed** estimates.
  NHTSA uses multiple imputation for untested drivers. National 2024: 11,907 fatalities (30%).
  The raw FARS coding (`fars_2024`) only captures explicitly coded cases (~15%).
  **Use this table for published analysis** — it's what NHTSA publishes and media cite.

In [ ]:
nhtsa = con.execute("SELECT * FROM nhtsa_imputed_2024 ORDER BY pct_alcohol_impaired_2024 DESC").df()
print(f"{len(nhtsa)} states | National: {nhtsa['alcohol_impaired_fatalities_2024'].sum():,} fatalities")
nhtsa

---
## 8. FBI UCR DUI Arrests by State (2023)
Source: FBI Uniform Crime Reporting Program — ICPSR Study 39298.

- **URL:** https://www.icpsr.umich.edu/web/NACJD/studies/39298
- **File:** `data/raw/ICPSR_39298-V1.zip` → `DS0002/39298-0002-Data.tsv`
- **Method:** ⚠️ **Manual download** — requires free ICPSR account registration
- **License:** ICPSR Terms of Use — no redistribution of raw data; derivative analysis OK; cite source
- **Offense code:** 220 = Driving Under the Influence

### ⚠️ Important limitation
Not all law enforcement agencies report to UCR. State-level totals reflect **only reporting
agencies** and significantly undercount actual arrests. Coverage varies widely by state.
Use `reporting_population` to compute per-capita rates for fair comparison.

**Citation:** United States Dept. of Justice, FBI. UCR Program Data: Arrests by Age,
Sex, and Race, Summarized Yearly, 2023. ICPSR [distributor], 2026.

In [ ]:
arrests = con.execute("SELECT * FROM dui_arrests_2023 ORDER BY total_dui_arrests DESC").df()
print(f"{len(arrests)} states | Total reported DUI arrests: {arrests['total_dui_arrests'].sum():,}")
print(f"Reporting agencies: {arrests['reporting_agencies'].sum():,} | Reporting pop: {arrests['reporting_population'].sum():,}")
arrests

---
## 9. FARS Prior DWI & Crash Speed
Source: NHTSA FARS 2024 `vehicle.csv` — `PREV_DWI` and `VSPD_LIM`.
Key: what % of impaired fatal-crash drivers are repeat offenders?

In [ ]:
prior_dwi = con.execute("SELECT * FROM fars_prior_dwi_speed ORDER BY state_fips").df()
print(f"{len(prior_dwi)} states")
print(f"Median % impaired with prior DWI: {prior_dwi['pct_impaired_with_prior_dwi'].median():.1f}%")
prior_dwi[['state_fips','pct_impaired_with_prior_dwi','median_crash_speed_limit','pct_crashes_high_speed']].head(10)

---
## 10. Speed Limits & VMT
- IIHS max posted speed limits (rural interstates, Aug 2026)
- FHWA Highway Statistics 2022 — vehicle miles traveled

In [ ]:
speed_vmt = con.execute("SELECT * FROM speed_limits_vmt ORDER BY state_name").df()
print(f"{len(speed_vmt)} states")
print(f"Speed: {speed_vmt['max_speed_limit_mph'].min()}-{speed_vmt['max_speed_limit_mph'].max()} mph")
print(f"VMT: {speed_vmt['vmt_millions_2022'].min():,}-{speed_vmt['vmt_millions_2022'].max():,}M miles")
speed_vmt.head(10)

---
## 11. BAC Testing Rates by State (FARS 2024 Person File)

**Source:** NHTSA FARS 2024 — `person.csv`  
**URL:** https://static.nhtsa.gov/nhtsa/downloads/FARS/2024/National/FARS2024NationalCSV.zip  
**Script:** `scripts/compute_bac_testing_rates.py`  
**License:** Public domain (U.S. government work)  
**DuckDB table:** `fars_bac_testing_2024`

### Method
Extracted from the `ALC_STATUS` field in `person.csv`:
- Code 0 = "Test Not Given" (driver was not BAC-tested)
- Code 2 = "Test Given" (BAC result known)
- Code 8 = "Not Reported"
- Code 9 = "Unknown if Tested"

Filtered to **drivers only** (`PER_TYP = 1`), then aggregated by state:
- `pct_bac_known_killed` — % of killed drivers with known BAC
- `pct_bac_known_all` — % of all drivers in fatal crashes with known BAC
- `pct_bac_known_surviving` — % of surviving drivers with known BAC
- Test type breakdown from `ATST_TYP`: blood (1), breath (2), vitreous (4), PBT (10)

### Key findings
- **National avg (killed drivers):** 66.8% have known BAC  
- **Range:** 9.6% (Mississippi) to 97.7% (Vermont)  
- States with mandatory coroner/ME testing laws average **78.2%** vs **51.7%** for probable-cause states  
- South Carolina: 76.4% (above average — their outlier status in NHTSA imputed numbers is NOT due to low testing)

In [ ]:
bac_testing = con.execute("SELECT * FROM fars_bac_testing_2024 ORDER BY pct_bac_known_killed DESC").df()
print(f"{len(bac_testing)} states")
print(f"Avg killed-driver BAC known: {bac_testing['pct_bac_known_killed'].mean():.1f}%")
print(f"Range: {bac_testing['pct_bac_known_killed'].min():.1f}% – {bac_testing['pct_bac_known_killed'].max():.1f}%")
bac_testing[['state_fips','total_drivers','pct_bac_known_killed','pct_bac_known_all','pct_bac_known_surviving','pct_blood_test']]

---
## 12. Mandatory BAC Testing Laws (Hand-Curated)

**Source:** Hand-curated from NHTSA Casanova et al. 2012 (DOT HS 811 661) + current state statutes  
**Reference:** https://rosap.ntl.bts.gov/view/dot/1940  
**File:** `data/raw/bac_testing_laws.csv`  
**License:** Public domain (legislative references)  
**DuckDB table:** `bac_testing_laws`

### Method
The NHTSA Casanova 2012 report documented that 25 states required BAC testing of all
(or nearly all) fatally injured drivers. We updated this with current statute research,
identifying 29 states with mandatory testing laws (4 enacted since 2012).

Each state classified by:
- `mandatory_testing_law` — yes/no
- `testing_scope` — all_fatally_injured, probable_cause, serious_injury_or_fatal
- `testing_authority` — coroner, medical_examiner, law_enforcement
- `no_refusal_program` — yes/no (whether state has/uses no-refusal warrants)
- `statute_citation` — specific state code reference

### Key findings
- **29 states** have mandatory BAC testing of fatally injured drivers (coroner/ME statute)
- **22 states** are probable-cause only
- Mandatory states average **78.2%** killed-driver BAC known vs **51.7%** for probable-cause
- Notable exceptions: CA, ID, OK have mandates but <60% compliance
- RI has no mandate but >91% testing (strong ME office practice)

In [ ]:
bac_laws = con.execute("SELECT * FROM bac_testing_laws ORDER BY state_fips").df()
print(f"{len(bac_laws)} states")
print(f"Mandatory: {(bac_laws['mandatory_testing_law'] == 'yes').sum()} | Probable-cause: {(bac_laws['mandatory_testing_law'] == 'no').sum()}")
bac_laws[['state_fips','state_abbr','state_name','mandatory_testing_law','testing_scope','testing_authority','statute_citation']]

---
## 13. NASID — State DUI Enforcement Procedures

**Source:** National Alliance to Stop Impaired Driving (NASID)  
**URL:** https://nasid.org/state/{state-slug}/  
**Script:** `scripts/scrape_nasid.py`  
**License:** Public reference (data sourced from NHTSA/FARS, May 2024)  
**DuckDB tables:** `nasid_enforcement` (raw), `nasid_enforcement_clean` (standardized)

### Method
Scraped all 51 state pages from nasid.org with 1.5s polite rate limiting.
Each page has a standardized set of enforcement law fields. Extracted 12 fields per state:

| Field | Description | Distribution |
|-------|-------------|-------------|
| Sobriety Checkpoints | Permitted/Prohibited | 39 permitted, 9 prohibited, 2 no authority, 1 alternative |
| No Refusal Programs | Active/Authorized/Not authorized | 9 active, 22 authorized, 20 not authorized |
| PBT Laws | Statute permits/Not explicit | 33 authorized, 18 not explicit |
| Ignition Interlocks | Mandatory all/High-BAC+repeat/etc. | 32 all-offender, 10 high-BAC+repeat, 4 discretionary |
| Felony DUI | 2nd/3rd/4th offense threshold | 26 third, 18 fourth, 4 second, 3 none |
| DUI Look-back | Years prior offenses count | 30 ten-year, 6 lifetime, 5 seven-year |
| High-BAC Threshold | Enhanced penalty trigger | 27 at 0.15, 9 at 0.16, 4 none |
| Open Container | Federal compliance | 40 compliant, 11 non-compliant |
| Testing Methods | Blood/Breath/Urine/Oral Fluid | 14 states allow oral fluid |
| ALS/ALR | Administrative license suspension | All 51 enacted |
| Social Host Laws | Alcohol/drugs | Varies |
| ALR Hardship | Hardship license availability | Varies |

### Data quality note
NASID sources their data from NHTSA/FARS (last updated May 2024). The site is maintained
by a coalition of impaired-driving organizations. Values represent statute-level law,
not necessarily enforcement practice.

In [ ]:
nasid = con.execute("SELECT * FROM nasid_enforcement ORDER BY state_fips").df()
print(f"{len(nasid)} states × {len(nasid.columns)} cols")
print(f"Columns: {list(nasid.columns)}")
nasid[['state_abbr','sobriety_checkpoints','no_refusal_programs','ignition_interlocks','roadside_preliminary_breath_test_pbt_laws']].head(10)

---
## Re-running ingestion

The pipeline uses multiple scripts for different data sources:

```bash
cd /path/to/dui-by-state

# Core ingestion (state ref, pop, FARS, NCSL, penalties, speed/VMT)
/opt/anaconda3/envs/data_projects/bin/python scripts/ingest_all.py

# BAC testing rates from FARS person.csv
/opt/anaconda3/envs/data_projects/bin/python scripts/compute_bac_testing_rates.py

# NASID enforcement scrape (all 51 state pages)
/opt/anaconda3/envs/data_projects/bin/python scripts/scrape_nasid.py
```

**Cached downloads:** `ingest_all.py` uses `fetch_cached()` — files on disk won't
be re-downloaded. Delete a specific file from `data/raw/` to force re-fetch.

**Manual data:** `data/raw/bac_testing_laws.csv` is hand-curated from statute
research — update manually if state laws change.

---
**Next:** open `02-clean.ipynb` for data cleaning and quality checks.

In [ ]:
con.close()